# TASK 06A - CNN From Scratch V2: Universal Monitoring Standard

Notebook ini adalah implementasi pertama standar monitoring, checkpoint, evaluasi validation, standalone competition inference, dan Kaggle submission per model. Semua tahap bersifat manual opt-in. Tidak ada training, validation evaluation, internal-test evaluation, inference kompetisi, atau CSV submission berjalan secara otomatis.

## Workflow Wajib

1. Manual Training
2. Validation Evaluation dengan `best.pt`
3. Training History dan Learning Curves
4. Classification Report
5. Confusion Matrix
6. Standalone Competition Test Inference
7. Kaggle Submission CSV
8. CSV Verification

Validation (383 gambar) digunakan untuk selection. Internal test (191 gambar) tidak dipakai pada notebook ini. Competition test resmi (480 gambar) hanya digunakan pada section submission.

In [10]:
from __future__ import annotations

import json
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from torch import nn


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("Project root tidak ditemukan. Buka notebook dari root proyek atau notebooks/.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.callbacks.early_stopping import EarlyStopping
from src.callbacks.lr_scheduler import build_scheduler
from src.callbacks.training_manager import TrainingManager
from src.data.dataloader import create_dataloader
from src.data.preprocessing import build_transforms, load_active_class_mapping, resolve_model_config
from src.evaluation.training_monitoring import history_review, load_training_history
from src.evaluation.visualization import plot_learning_curves
from src.inference.competition import CompetitionTestDataset, audit_competition_test, create_competition_dataloader, load_submission_template
from src.inference.predict import load_cnn_best_checkpoint, predict_torch_classifier
from src.inference.submission import validate_submission_csv, write_submission_csv
from src.models import CNNFromScratch
from src.training.validate import validate_one_epoch

print(f"Project root: {PROJECT_ROOT}")
print(f"CUDA tersedia: {torch.cuda.is_available()}")


Project root: D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification
CUDA tersedia: True


## Experiment Configuration

Konfigurasi V2 memakai preprocessing CNN yang sama dengan baseline untuk perbandingan yang terkendali. Ubah hyperparameter hanya sebelum training manual dan gunakan experiment ID baru untuk eksperimen baru agar artefak tidak tertimpa.

In [17]:
SEED = 42
EXPERIMENT_ID = "cnn_scratch_v2"
MODEL_TYPE = "cnn_v2"
RUN_TRAINING = False
RUN_VALIDATION_EVALUATION = True
RUN_TEST_INFERENCE = False

CONFIG = {
    "image_size": (224, 224),
    "batch_size": 16,
    "max_epochs": 100,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "dropout": 0.40,
    "num_workers": 0,
    "augmentation_enabled": True,
    "amp_enabled": torch.cuda.is_available(),
    "gradient_accumulation_steps": 1,
    "gradient_clip_norm": 1.0,
}
EARLY_STOPPING_CONFIG = {"monitor": "val_loss", "mode": "min", "patience": 10, "min_delta": 0.001}
SCHEDULER_CONFIG = {"name": "reduce_on_plateau", "monitor": "val_loss", "mode": "min", "factor": 0.5, "patience": 4, "min_lr": 1e-6}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print({"experiment_id": EXPERIMENT_ID, "device": str(DEVICE), **CONFIG})


{'experiment_id': 'cnn_scratch_v2', 'device': 'cuda', 'image_size': (224, 224), 'batch_size': 16, 'max_epochs': 100, 'learning_rate': 0.001, 'weight_decay': 0.0001, 'dropout': 0.4, 'num_workers': 0, 'augmentation_enabled': True, 'amp_enabled': True, 'gradient_accumulation_steps': 1, 'gradient_clip_norm': 1.0}


## Data, Preprocessing, and Loaders

Only the active V1 final manifest is used. This cell rejects forbidden sources and creates train/validation loaders. No internal-test loader is created.

In [12]:
class_mapping = load_active_class_mapping(PROJECT_ROOT)
class_names = tuple(name for name, _ in sorted(class_mapping.items(), key=lambda item: item[1]))
if len(class_mapping) != 8:
    raise ValueError("Expected eight active waste classes.")

cnn_v2_preprocessing = resolve_model_config(MODEL_TYPE, project_root=PROJECT_ROOT)
if tuple(cnn_v2_preprocessing["image_size"]) != CONFIG["image_size"] or cnn_v2_preprocessing["normalization"] is not None:
    raise ValueError("CNN V2 preprocessing must remain RGB 224x224 with no additional normalization.")

train_loader = create_dataloader("train", MODEL_TYPE, batch_size=CONFIG["batch_size"], seed=SEED, num_workers=CONFIG["num_workers"], pin_memory=DEVICE.type == "cuda", project_root=PROJECT_ROOT, augmentation_enabled=True)
val_loader = create_dataloader("val", MODEL_TYPE, batch_size=CONFIG["batch_size"], seed=SEED, num_workers=CONFIG["num_workers"], pin_memory=DEVICE.type == "cuda", project_root=PROJECT_ROOT, augmentation_enabled=False)
print({"classes": class_mapping, "train_samples": len(train_loader.dataset), "validation_samples": len(val_loader.dataset)})


{'classes': {'cardboard': 0, 'electronic': 1, 'foodwaste': 2, 'glass': 3, 'metal': 4, 'paper': 5, 'plastic': 6, 'textile': 7}, 'train_samples': 1340, 'validation_samples': 383}


## Model, Callbacks, and Persistent Monitoring

`TrainingManager` emits one tqdm batch bar with smoothed loss, running accuracy, and LR. After every epoch it atomically updates JSON, CSV, and JSONL history, `last.pt`, and conditionally `best.pt`, then prints an epoch summary. Resume must be activated manually and starts only after persisted epochs.

In [13]:
model = CNNFromScratch(num_classes=len(class_mapping), dropout=CONFIG["dropout"])
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler, scheduler_kind = build_scheduler(optimizer, SCHEDULER_CONFIG)
manager = TrainingManager(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    class_mapping=class_mapping,
    model_identifier="cnn_from_scratch",
    experiment_id=EXPERIMENT_ID,
    project_root=PROJECT_ROOT,
    device=DEVICE,
    scheduler=scheduler,
    scheduler_kind=scheduler_kind,
    early_stopping=EarlyStopping(**EARLY_STOPPING_CONFIG),
    training_config={**CONFIG, "scheduler": SCHEDULER_CONFIG, "early_stopping": EARLY_STOPPING_CONFIG, "model_type": MODEL_TYPE},
    seed=SEED,
)
print({"parameters": sum(parameter.numel() for parameter in model.parameters()), "history_json": str(manager.metrics_path), "history_csv": str(manager.metrics_csv_path), "checkpoint_dir": str(manager.checkpoints.directory)})


{'parameters': 892648, 'history_json': 'D:\\FILE KULIAH\\Lomba Gelar 2026\\DSC_2026_Waste_Classification\\outputs\\metrics\\cnn_scratch_v2.json', 'history_csv': 'D:\\FILE KULIAH\\Lomba Gelar 2026\\DSC_2026_Waste_Classification\\outputs\\metrics\\cnn_scratch_v2.csv', 'checkpoint_dir': 'D:\\FILE KULIAH\\Lomba Gelar 2026\\DSC_2026_Waste_Classification\\models\\checkpoints\\cnn_scratch_v2'}


## 1. Manual Training

Set `RUN_TRAINING=True` only after reviewing configuration and GPU memory. Batch monitoring includes `Epoch n/max`, `batch/total`, smoothed `loss`, running `acc`, LR, tqdm elapsed/ETA. The epoch summary records train/validation metrics, macro precision/recall/F1, LR change, duration, best epoch/loss, early stopping counter, and checkpoint status.

In [14]:
if RUN_TRAINING:
    history = manager.fit(
        train_loader,
        val_loader,
        max_epochs=CONFIG["max_epochs"],
        amp_enabled=CONFIG["amp_enabled"],
        gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
        gradient_clip_norm=CONFIG["gradient_clip_norm"],
        show_progress=True,
    )
    print(f"Training selesai/berhenti setelah {len(history)} epoch. History tersimpan aman pada {manager.metrics_path} dan {manager.metrics_csv_path}.")
else:
    print("Training tidak dijalankan. Set RUN_TRAINING=True untuk memulai manual training.")


Epoch 1/100 train:   0%|          | 0/84 [00:00<?, ?it/s]

Epoch 1/100 | train_loss=1.8459 train_acc=30.60% | val_loss=1.7342 val_acc=35.25% | macro_f1=0.2953 lr=1.00e-03 | 33.8s | best_epoch=1 best_val_loss=1.7342 | early_stop=0 | BEST SAVED


Epoch 2/100 train:   0%|          | 0/84 [00:00<?, ?it/s]

Epoch 2/100 | train_loss=1.7063 train_acc=37.39% | val_loss=1.6902 val_acc=38.38% | macro_f1=0.3542 lr=1.00e-03 | 10.0s | best_epoch=2 best_val_loss=1.6902 | early_stop=0 | BEST SAVED


Epoch 3/100 train:   0%|          | 0/84 [00:00<?, ?it/s]

Epoch 3/100 | train_loss=1.6507 train_acc=39.25% | val_loss=1.6110 val_acc=42.56% | macro_f1=0.3928 lr=1.00e-03 | 9.7s | best_epoch=3 best_val_loss=1.6110 | early_stop=0 | BEST SAVED


... (148 outputs left out)

Epoch 78/100 train:   0%|          | 0/84 [00:00<?, ?it/s]

Epoch 78/100 | train_loss=0.9467 train_acc=68.96% | val_loss=1.2628 val_acc=60.31% | macro_f1=0.5982 lr=6.25e-05 | 9.7s | best_epoch=69 best_val_loss=1.2411 | early_stop=9 | LAST UPDATED


Epoch 79/100 train:   0%|          | 0/84 [00:00<?, ?it/s]

Epoch 79/100 | train_loss=0.9700 train_acc=67.24% | val_loss=1.2460 val_acc=60.05% | macro_f1=0.5947 lr=3.13e-05 | 9.7s | best_epoch=69 best_val_loss=1.2411 | early_stop=10 | LAST UPDATED
Training selesai/berhenti setelah 79 epoch. History tersimpan aman pada D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\outputs\metrics\cnn_scratch_v2.json dan D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\outputs\metrics\cnn_scratch_v2.csv.


## 2-3. Persistent History and Learning Curves

History dapat dimuat ulang setelah kernel restart. Tabel berikut menampilkan semua epoch, lima `val_loss` terbaik, epoch terakhir, perubahan LR, dan status early stopping. Grafik disimpan berdasarkan experiment ID dan ditampilkan dengan backend notebook-compatible.

In [15]:
metrics_path = PROJECT_ROOT / "outputs" / "metrics" / f"{EXPERIMENT_ID}.json"
if metrics_path.is_file():
    actual_history = load_training_history(metrics_path)
    review = history_review(actual_history)
    display(review["all_epochs"])
    display(review["best_five_by_val_loss"])
    display(review["last_epoch"])
    display(review["learning_rate_changes"])
    display(review["early_stopping"])

    curve_path = PROJECT_ROOT / "outputs" / "figures" / f"{EXPERIMENT_ID}_learning_curves.png"
    plot_learning_curves(actual_history, curve_path, title=f"CNN From Scratch V2 - {EXPERIMENT_ID}", display_figure=True)
    print(f"Learning curves tersimpan: {curve_path}")
else:
    print(f"History belum tersedia: {metrics_path}. Jalankan manual training terlebih dahulu.")


,experiment_id,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1,learning_rates,...,learning_rate_changed,epoch_seconds,best_epoch,best_validation_loss,early_stopping_counter,early_stopping,early_stopping_reason,checkpoint_best_saved,checkpoint_last,checkpoint_status
0,cnn_scratch_v2,0,1.845932,0.305970,1.734201,0.352480,0.352999,0.345987,0.295340,[0.001],...,False,33.843297,0,1.734201,0,False,NaN,True,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,BEST SAVED
1,cnn_scratch_v2,1,1.706329,0.373881,1.690219,0.383812,0.384027,0.381998,0.354210,[0.001],...,False,10.024417,1,1.690219,0,False,NaN,True,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,BEST SAVED
2,cnn_scratch_v2,2,1.650674,0.392537,1.611032,0.425587,0.414265,0.431822,0.392813,[0.001],...,False,9.665322,2,1.611032,0,False,NaN,True,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,BEST SAVED
3,cnn_scratch_v2,3,1.645777,0.395522,1.625961,0.417755,0.416540,0.421018,0.386751,[0.001],...,False,9.570700,2,1.611032,1,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
4,cnn_scratch_v2,4,1.649749,0.395522,1.770740,0.326371,0.327332,0.335657,0.298018,[0.001],...,False,9.620537,2,1.611032,2,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
74,cnn_scratch_v2,74,0.954920,0.673134,1.243360,0.610966,0.625900,0.608915,0.609421,[0.000063],...,False,9.760289,68,1.241106,6,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
75,cnn_scratch_v2,75,0.945022,0.676866,1.273466,0.597911,0.615953,0.599527,0.598337,[0.000063],...,False,9.737850,68,1.241106,7,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
76,cnn_scratch_v2,76,0.941312,0.682836,1.246813,0.624021,0.631578,0.623148,0.621922,[0.000063],...,False,9.666481,68,1.241106,8,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
77,cnn_scratch_v2,77,0.946738,0.689552,1.262805,0.603133,0.605893,0.601781,0.598176,[0.000063],...,False,9.697552,68,1.241106,9,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED


,experiment_id,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1,learning_rates,...,learning_rate_changed,epoch_seconds,best_epoch,best_validation_loss,early_stopping_counter,early_stopping,early_stopping_reason,checkpoint_best_saved,checkpoint_last,checkpoint_status
68,cnn_scratch_v2,68,1.037323,0.635821,1.241106,0.600522,0.601030,0.601239,0.595862,[0.000125],...,False,9.624705,68,1.241106,0,False,NaN,True,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,BEST SAVED
74,cnn_scratch_v2,74,0.954920,0.673134,1.243360,0.610966,0.625900,0.608915,0.609421,[0.000063],...,False,9.760289,68,1.241106,6,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
78,cnn_scratch_v2,78,0.970035,0.672388,1.246016,0.600522,0.598870,0.601252,0.594696,[0.000031],...,True,9.677030,68,1.241106,10,True,val_loss did not improve by 0.001 for 10 epochs,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
76,cnn_scratch_v2,76,0.941312,0.682836,1.246813,0.624021,0.631578,0.623148,0.621922,[0.000063],...,False,9.666481,68,1.241106,8,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
66,cnn_scratch_v2,66,1.019046,0.651493,1.253551,0.603133,0.602011,0.604638,0.597649,[0.000125],...,False,9.672716,66,1.253551,0,False,NaN,True,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,BEST SAVED


,experiment_id,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1,learning_rates,...,learning_rate_changed,epoch_seconds,best_epoch,best_validation_loss,early_stopping_counter,early_stopping,early_stopping_reason,checkpoint_best_saved,checkpoint_last,checkpoint_status
78,cnn_scratch_v2,78,0.970035,0.672388,1.246016,0.600522,0.59887,0.601252,0.594696,[0.000031],...,True,9.67703,68,1.241106,10,True,val_loss did not improve by 0.001 for 10 epochs,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED


,experiment_id,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1,learning_rates,...,learning_rate_changed,epoch_seconds,best_epoch,best_validation_loss,early_stopping_counter,early_stopping,early_stopping_reason,checkpoint_best_saved,checkpoint_last,checkpoint_status
15,cnn_scratch_v2,15,1.420929,0.502239,1.587471,0.443864,0.459321,0.444149,0.414839,[0.0005],...,True,9.620844,10,1.470103,5,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
28,cnn_scratch_v2,28,1.275186,0.558209,1.471385,0.490862,0.481008,0.490962,0.476280,[0.00025],...,True,9.621902,23,1.425924,5,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
47,cnn_scratch_v2,47,1.107342,0.607463,1.323751,0.558747,0.564054,0.562014,0.555659,[0.000125],...,True,9.747923,42,1.314150,5,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
73,cnn_scratch_v2,73,0.974756,0.670149,1.301576,0.590078,0.608404,0.593085,0.591104,[0.000063],...,True,9.731215,68,1.241106,5,False,NaN,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED
78,cnn_scratch_v2,78,0.970035,0.672388,1.246016,0.600522,0.598870,0.601252,0.594696,[0.000031],...,True,9.677030,68,1.241106,10,True,val_loss did not improve by 0.001 for 10 epochs,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED


,experiment_id,epoch,train_loss,train_accuracy,val_loss,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1,learning_rates,...,learning_rate_changed,epoch_seconds,best_epoch,best_validation_loss,early_stopping_counter,early_stopping,early_stopping_reason,checkpoint_best_saved,checkpoint_last,checkpoint_status
78,cnn_scratch_v2,78,0.970035,0.672388,1.246016,0.600522,0.59887,0.601252,0.594696,[0.000031],...,True,9.67703,68,1.241106,10,True,val_loss did not improve by 0.001 for 10 epochs,False,D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...,LAST UPDATED


<Figure size 1300x900 with 4 Axes>

Learning curves tersimpan: D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\outputs\figures\cnn_scratch_v2_learning_curves.png


## 4-5. Validation Evaluation, Classification Report, and Confusion Matrix

Validation selalu memuat `best.pt`, bukan model epoch terakhir. Internal test tidak digunakan.

In [18]:
validation_metrics = None
CNN_V2_BEST_CHECKPOINT = PROJECT_ROOT / "models" / "checkpoints" / EXPERIMENT_ID / "best.pt"
if RUN_VALIDATION_EVALUATION:
    loaded_cnn_v2 = load_cnn_best_checkpoint(project_root=PROJECT_ROOT, checkpoint_path=CNN_V2_BEST_CHECKPOINT, device=DEVICE, expected_class_mapping=class_mapping)
    validation_metrics = validate_one_epoch(loaded_cnn_v2.model, val_loader, criterion, class_mapping, loaded_cnn_v2.device)
    display(pd.DataFrame(validation_metrics["classification_report"]).T)

    confusion = np.asarray(validation_metrics["confusion_matrix"])
    figure, axis = plt.subplots(figsize=(10, 8))
    sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names, ax=axis)
    axis.set_xlabel("Prediksi")
    axis.set_ylabel("Label Aktual")
    axis.set_title("CNN From Scratch V2 - Validation Confusion Matrix")
    display(figure)
    plt.close(figure)
else:
    print("Validation evaluation tidak dijalankan. Set RUN_VALIDATION_EVALUATION=True setelah best.pt tersedia.")


,precision,recall,f1-score,support
cardboard,0.597222,0.826923,0.693548,52.000000
electronic,0.805556,0.707317,0.753247,41.000000
foodwaste,0.695652,0.653061,0.673684,49.000000
glass,0.542857,0.463415,0.500000,41.000000
metal,0.447368,0.354167,0.395349,48.000000
paper,0.614035,0.777778,0.686275,45.000000
plastic,0.568966,0.515625,0.540984,64.000000
textile,0.536585,0.511628,0.523810,43.000000
accuracy,0.600522,0.600522,0.600522,0.600522
macro avg,0.601030,0.601239,0.595862,383.000000


<Figure size 1000x800 with 2 Axes>

## 6-8. Standalone Competition Test Inference, Submission, and CSV Verification

The next setup cell is self-contained and can run in a fresh kernel by itself. It resolves the project root, adds it to `sys.path` before importing `src`, reads model configuration/class mapping/checkpoint metadata from disk, audits 480 official test images, and does not use values from training cells. Sample `waste_category` values are never used as answers.

In [20]:
# Standalone inference setup; safe to run first in a fresh kernel.
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display


def find_inference_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise RuntimeError("Project root tidak ditemukan. Buka notebook dari root proyek atau notebooks/.")


INFERENCE_PROJECT_ROOT = find_inference_project_root(Path.cwd())
if str(INFERENCE_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(INFERENCE_PROJECT_ROOT))

from src.callbacks.model_checkpoint import load_checkpoint
from src.data.preprocessing import build_transforms, load_active_class_mapping, resolve_model_config
from src.inference.competition import CompetitionTestDataset, audit_competition_test, create_competition_dataloader, load_submission_template
from src.inference.predict import load_cnn_best_checkpoint, predict_torch_classifier
from src.inference.submission import validate_submission_csv, write_submission_csv

INFERENCE_EXPERIMENT_ID = "cnn_scratch_v2"
INFERENCE_MODEL_TYPE = "cnn_v2"
INFERENCE_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INFERENCE_CONFIG_PATH = INFERENCE_PROJECT_ROOT / "configs" / "models" / "cnn_v2.yaml"
INFERENCE_CHECKPOINT_PATH = INFERENCE_PROJECT_ROOT / "models" / "checkpoints" / INFERENCE_EXPERIMENT_ID / "best.pt"
INFERENCE_OUTPUT_PATH = INFERENCE_PROJECT_ROOT / "Final Dataset for Submission to Kaggle" / "submission_cnn_v2.csv"
INFERENCE_TEST_DIR = INFERENCE_PROJECT_ROOT / "data" / "competition" / "test"
INFERENCE_SAMPLE_PATH = INFERENCE_PROJECT_ROOT / "data" / "competition" / "SampleSubmission_DSC.csv"

for required_path in (INFERENCE_CONFIG_PATH, INFERENCE_SAMPLE_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(f"Artefak inference tidak ditemukan: {required_path}")
if not INFERENCE_TEST_DIR.is_dir():
    raise FileNotFoundError(f"Test resmi tidak ditemukan: {INFERENCE_TEST_DIR}")

INFERENCE_PREPROCESSING = resolve_model_config(INFERENCE_MODEL_TYPE, project_root=INFERENCE_PROJECT_ROOT)
INFERENCE_TRANSFORM = build_transforms(INFERENCE_MODEL_TYPE, "inference", project_root=INFERENCE_PROJECT_ROOT)
INFERENCE_VALIDATION_TRANSFORM = build_transforms(INFERENCE_MODEL_TYPE, "val", project_root=INFERENCE_PROJECT_ROOT)
if INFERENCE_TRANSFORM.config != INFERENCE_VALIDATION_TRANSFORM.config or INFERENCE_TRANSFORM.augmentation is not None:
    raise ValueError("Inference preprocessing must match validation without random augmentation.")
if tuple(INFERENCE_PREPROCESSING["image_size"]) != (224, 224) or INFERENCE_PREPROCESSING["normalization"] is not None:
    raise ValueError("CNN V2 preprocessing does not match its configured validation preprocessing.")

INFERENCE_MAPPING = load_active_class_mapping(INFERENCE_PROJECT_ROOT)
if len(INFERENCE_MAPPING) != 8:
    raise ValueError("Expected exactly eight active classes.")
INFERENCE_TEMPLATE = load_submission_template(INFERENCE_SAMPLE_PATH)
INFERENCE_AUDIT = audit_competition_test(project_root=INFERENCE_PROJECT_ROOT, test_dir=INFERENCE_TEST_DIR, sample_submission_path=INFERENCE_SAMPLE_PATH, expected_count=480)

if INFERENCE_CHECKPOINT_PATH.is_file():
    INFERENCE_CHECKPOINT_METADATA = load_checkpoint(INFERENCE_CHECKPOINT_PATH, map_location="cpu")
    if dict(INFERENCE_CHECKPOINT_METADATA.get("class_mapping", {})) != INFERENCE_MAPPING:
        raise ValueError("Checkpoint class mapping does not match the active training mapping.")
    INFERENCE_BATCH_SIZE = int(INFERENCE_CHECKPOINT_METADATA.get("training_config", {}).get("batch_size", 16))
    checkpoint_status = "available"
else:
    INFERENCE_CHECKPOINT_METADATA = None
    INFERENCE_BATCH_SIZE = 16
    checkpoint_status = "not yet available"

display(pd.Series({
    "project_root": str(INFERENCE_PROJECT_ROOT),
    "experiment_id": INFERENCE_EXPERIMENT_ID,
    "device": str(INFERENCE_DEVICE),
    "checkpoint_status": checkpoint_status,
    "batch_size": INFERENCE_BATCH_SIZE,
    "test_images": INFERENCE_AUDIT.image_count,
    "sample_rows": INFERENCE_AUDIT.sample_row_count,
    "output": str(INFERENCE_OUTPUT_PATH),
}))
print("Standalone setup selesai. Tidak ada training, inference, atau CSV submission dijalankan pada cell ini.")


project_root         D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...
experiment_id                                           cnn_scratch_v2
device                                                            cuda
checkpoint_status                                            available
batch_size                                                          16
test_images                                                        480
sample_rows                                                        480
output               D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste...
dtype: object

Standalone setup selesai. Tidak ada training, inference, atau CSV submission dijalankan pada cell ini.


In [21]:
# Manual opt-in inference; run only after the standalone setup cell.
RUN_TEST_INFERENCE = True

if RUN_TEST_INFERENCE:
    if not INFERENCE_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f"Best checkpoint belum tersedia: {INFERENCE_CHECKPOINT_PATH}")
    inference_dataset = CompetitionTestDataset(model_type=INFERENCE_MODEL_TYPE, project_root=INFERENCE_PROJECT_ROOT, test_dir=INFERENCE_TEST_DIR, sample_submission_path=INFERENCE_SAMPLE_PATH, transform=INFERENCE_TRANSFORM, expected_count=480)
    inference_loader = create_competition_dataloader(inference_dataset, batch_size=INFERENCE_BATCH_SIZE, num_workers=0, pin_memory=INFERENCE_DEVICE.type == "cuda")
    inference_model = load_cnn_best_checkpoint(project_root=INFERENCE_PROJECT_ROOT, checkpoint_path=INFERENCE_CHECKPOINT_PATH, device=INFERENCE_DEVICE, expected_class_mapping=INFERENCE_MAPPING)
    if inference_model.model.training:
        raise RuntimeError("Loaded model must be in eval mode for inference.")
    predictions = predict_torch_classifier(inference_model.model, inference_loader, index_to_class=inference_model.index_to_class, device=inference_model.device)
    submission_path = write_submission_csv(INFERENCE_TEMPLATE, predictions, INFERENCE_OUTPUT_PATH, allowed_classes=inference_model.index_to_class, overwrite=False)
    validate_submission_csv(submission_path, INFERENCE_TEMPLATE, allowed_classes=inference_model.index_to_class)
    submission_frame = pd.read_csv(submission_path, dtype={"image_id": str})
    display(submission_frame.head())
    display(submission_frame["waste_category"].value_counts().rename("prediction_count"))
    print(f"Submission valid tersimpan: {submission_path}")
else:
    print("Competition inference tidak dijalankan. Set RUN_TEST_INFERENCE=True setelah standalone setup dan best.pt tersedia.")


,image_id,waste_category
0,001,textile
1,002,foodwaste
2,003,foodwaste
3,004,metal
4,005,paper


waste_category
plastic       88
cardboard     70
electronic    67
foodwaste     66
metal         61
paper         60
glass         37
textile       31
Name: prediction_count, dtype: int64

Submission valid tersimpan: D:\FILE KULIAH\Lomba Gelar 2026\DSC_2026_Waste_Classification\Final Dataset for Submission to Kaggle\submission_cnn_v2.csv


In [22]:
# Optional CSV verification; no training or inference.
if INFERENCE_OUTPUT_PATH.is_file():
    validate_submission_csv(INFERENCE_OUTPUT_PATH, INFERENCE_TEMPLATE, allowed_classes=INFERENCE_MAPPING)
    verified_submission = pd.read_csv(INFERENCE_OUTPUT_PATH, dtype={"image_id": str})
    display(verified_submission.head())
    print(f"CSV verification selesai: {len(verified_submission)} baris valid.")
else:
    print(f"CSV belum tersedia: {INFERENCE_OUTPUT_PATH}")


,image_id,waste_category
0,001,textile
1,002,foodwaste
2,003,foodwaste
3,004,metal
4,005,paper


CSV verification selesai: 480 baris valid.


## Standard Handoff

Notebook ini adalah implementasi pertama standar proyek. MobileNetV3, EfficientNet, ResNet50, dan YOLOv8 harus memakai history, visualisasi, checkpoint selection, standalone inference, CSV validation, dan naming per-model yang sama. Preprocessing/adapter model wajib tetap spesifik arsitektur; YOLOv8 memakai adapter logging/checkpoint sendiri namun harus menghasilkan schema history dan submission yang kompatibel.